In [1]:
!pip install rasterio
from google.colab import files
import os
os.makedirs('/content/raw_tifs', exist_ok=True)
uploaded = files.upload()  # select all dfw_aod_*.tif
for name, data in uploaded.items():
    open(f'/content/raw_tifs/{name}', 'wb').write(data)
print('uploaded', len(uploaded))

Saving dfw_aod_2023-06-01.tif to dfw_aod_2023-06-01.tif
Saving dfw_aod_2023-06-02.tif to dfw_aod_2023-06-02.tif
Saving dfw_aod_2023-06-04.tif to dfw_aod_2023-06-04.tif
Saving dfw_aod_2023-06-05.tif to dfw_aod_2023-06-05.tif
Saving dfw_aod_2023-06-06.tif to dfw_aod_2023-06-06.tif
Saving dfw_aod_2023-06-07.tif to dfw_aod_2023-06-07.tif
Saving dfw_aod_2023-06-08.tif to dfw_aod_2023-06-08.tif
Saving dfw_aod_2023-06-09.tif to dfw_aod_2023-06-09.tif
Saving dfw_aod_2023-06-10.tif to dfw_aod_2023-06-10.tif
Saving dfw_aod_2023-06-11.tif to dfw_aod_2023-06-11.tif
Saving dfw_aod_2023-06-12.tif to dfw_aod_2023-06-12.tif
Saving dfw_aod_2023-06-13.tif to dfw_aod_2023-06-13.tif
Saving dfw_aod_2023-06-14.tif to dfw_aod_2023-06-14.tif
Saving dfw_aod_2023-06-15.tif to dfw_aod_2023-06-15.tif
Saving dfw_aod_2023-06-16.tif to dfw_aod_2023-06-16.tif
Saving dfw_aod_2023-06-17.tif to dfw_aod_2023-06-17.tif
Saving dfw_aod_2023-06-18.tif to dfw_aod_2023-06-18.tif
Saving dfw_aod_2023-06-19.tif to dfw_aod_2023-06

In [5]:
from pathlib import Path
import numpy as np
import rasterio

RAW = Path('/content/raw_tifs')
OUT = Path('/content/sprint2_patches.npz')
PATCH, SCALE, MIN_VALID = 64, 4, 0.6

def box_average(hr, mask):
    n, _, h, w = hr.shape
    hs, ws = h // SCALE, w // SCALE
    hr = hr.reshape(n, 1, hs, SCALE, ws, SCALE)
    mask = mask.reshape(n, 1, hs, SCALE, ws, SCALE)
    count = mask.sum(axis=(3, 5))
    lr = (hr * mask).sum(axis=(3, 5)) / np.maximum(count, 1.0)
    lr[count == 0] = 0.0
    return lr.astype(np.float32)

tifs = sorted(RAW.glob('*.tif'))
print('tifs found:', len(tifs))
patches, masks = [], []
for path in tifs:
    with rasterio.open(path) as src:
        aod = src.read(1).astype(np.float32)
    valid = np.isfinite(aod) & (aod > -0.1) & (aod < 5)
    aod = np.where(valid, aod, 0.0)
    h, w = aod.shape
    for r in range(0, h - PATCH + 1, PATCH // 2):
        for c in range(0, w - PATCH + 1, PATCH // 2):
            hr = aod[r:r+PATCH, c:c+PATCH]
            m = valid[r:r+PATCH, c:c+PATCH]
            if m.mean() < MIN_VALID:
                continue
            patches.append(hr)
            masks.append(m.astype(np.float32))

if not patches:
    raise SystemExit('No patch passed the 60% valid cut.')
hr = np.stack(patches)[:, None]
mask = np.stack(masks)[:, None]
lr = box_average(hr, mask)
finite = hr[mask > 0]
print('patches:', hr.shape[0])
print('HR min/median/max:', float(finite.min()), float(np.median(finite)), float(finite.max()))
np.savez_compressed(OUT, lr=lr, hr=hr, mask=mask, scale=SCALE)
print('saved', OUT)

from google.colab import files
files.download(str(OUT))

tifs found: 29
patches: 144
HR min/median/max: 0.0 0.1445000022649765 0.8009999990463257
saved /content/sprint2_patches.npz


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>